# Spick Daten einlesen

### Libraries & Settings

In [1]:
# Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, text
import matplotlib.pyplot as plt

# Settings
import warnings
warnings.filterwarnings("ignore")

# Define settings for graphics
plt.style.use('dark_background')

# Current working directory
print(os.getcwd())

/workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01


### Read Data to Dataframe
#### Nicht ausführbar, da die Dateien fehlen

In [2]:
#Einzeln:
# Read apartment data
df = pd.read_csv('Data/apartments_data_prepared.csv', sep=',')
df.head(5)


#Ausführlicher:

# Define path to data
path = 'Data/'

# Read data from .csv files
df_customers = pd.read_csv(f'{path}Customers.csv', sep=',')
df_categories = pd.read_csv(f'{path}Categories.csv', sep=',')
df_employees = pd.read_csv(f'{path}Employees.csv', sep=',')
df_order_details = pd.read_csv(f'{path}OrderDetails.csv', sep=',')
df_orders = pd.read_csv(f'{path}Orders.csv', sep=',')
df_products = pd.read_csv(f'{path}Products.csv', sep=',')
df_shippers = pd.read_csv(f'{path}Shippers.csv', sep=',')
df_suppliers = pd.read_csv(f'{path}Suppliers.csv', sep=',')

# Change column names to lowercase
df_customers.columns = df_customers.columns.str.lower()
df_categories.columns = df_categories.columns.str.lower()
df_employees.columns = df_employees.columns.str.lower()
df_order_details.columns = df_order_details.columns.str.lower()
df_orders.columns = df_orders.columns.str.lower()
df_products.columns = df_products.columns.str.lower()
df_shippers.columns = df_shippers.columns.str.lower()
df_suppliers.columns = df_suppliers.columns.str.lower()

# Check dimensions of DataFrames
print("Dimensions of DataFrames:")
print(f"Customers: {df_customers.shape}")
print(f"Categories: {df_categories.shape}")
print(f"Employees: {df_employees.shape}")
print(f"OrderDetails: {df_order_details.shape}")
print(f"Orders: {df_orders.shape}")
print(f"Products: {df_products.shape}")
print(f"Shippers: {df_shippers.shape}")
print(f"Suppliers: {df_suppliers.shape}")

FileNotFoundError: [Errno 2] No such file or directory: 'Data/apartments_data_prepared.csv'

### Create database connection

In [ ]:
# Set up database connection
user = "pgadmin"
password = "geheim"
host = "localhost"
port = "5432"
database = "northwind"

# Create Connection URL
db_connection_url = "postgresql://" + user + ":" + password +\
                    "@" + host + ":" + port + "/" + database

# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Test database connection
with engine.connect() as connection:
    result = connection.execute(text('SELECT current_database()'))
    print(result.fetchone())

# Dispose the engine
engine.dispose()

### Write data frames to database tables

In [ ]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Drop tables if it exist
with engine.connect() as connection:
    connection.execute(text("DROP TABLE IF EXISTS public.categories CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.customers CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.employees CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.orderdetails CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.orders CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.products CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.shippers CASCADE;"))
    connection.execute(text("DROP TABLE IF EXISTS public.suppliers CASCADE;"))
    connection.commit()

# Write DataFrames to database
df_categories.to_sql('categories', engine)
df_customers.to_sql('customers', engine)
df_employees.to_sql('employees', engine)
df_order_details.to_sql('orderdetails', engine)
df_orders.to_sql('orders', engine)
df_products.to_sql('products', engine)
df_shippers.to_sql('shippers', engine)
df_suppliers.to_sql('suppliers', engine)

# Dispose the engine
engine.dispose()

### List tables in the database

In [ ]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Open a connection
with engine.connect() as connection:

    # Execute the query
    result = connection.execute(text("""SELECT table_name
                                        FROM information_schema.tables
                                        WHERE table_schema = 'public';"""))
    
    # Fetch and print the results
    for row in result:
        print(row[0])

# Dispose the engine
engine.dispose()

### Create a SQL query to retrieve product information from a database.

In [ ]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)

# Write data to table
df_sub = pd.read_sql_query('''SELECT 
                                Products.ProductID, 
                                Products.ProductName, 
                                Categories.CategoryName,
                                Products.Price
                             FROM Products
                             INNER JOIN Categories ON Products.CategoryID = Categories.CategoryID
                             ORDER BY Products.ProductID;''', 
                          con=engine)

# Dispose the engine
engine.dispose()

# Show the data
df_sub

### Plot product prices

In [ ]:
# Plot Histogram
fig = plt.figure( figsize=(7,4))
plt.xticks(fontsize=14, rotation=0)
plt.yticks(fontsize=14, rotation=0)
n, bins, patches = plt.hist(x=df_sub['price'], 
                            bins=20, 
                            color='#C53115',
                            alpha=1.00, 
                            rwidth=0.95
                   )
plt.grid(alpha=0.25)
plt.ticklabel_format(style='plain')

# Set labels
plt.xlabel('Price (USD)', fontsize=10, labelpad=10)
plt.ylabel('Frequency', fontsize=10, labelpad=10)
plt.title('Histogram of product prices', fontsize=12, pad=10)

# Set fontsize of tick labels
plt.xticks(fontsize = 10)
plt.yticks(fontsize = 10)

plt.show()

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

# Beispielqueries

In [ ]:
1. What are the details of all customers whose country is 'Spain'?
SELECT * 
FROM Customers
WHERE Country = 'Spain';

2. What are the distinct cities of customers from Germany with a city containing the letter 'B'?
SELECT DISTINCT City 
FROM Customers 
WHERE Country = 'Germany' 
AND City LIKE '%B%';

3. What are the number of orders placed by each customer? Sort the result by the number of orders in descending order.
SELECT 
    CustomerID, 
    COUNT(OrderID) AS OrderCount
FROM Orders
GROUP BY CustomerID
ORDER BY OrderCount DESC;

4. What are the customers who have placed more than 3 orders?
SELECT 
    CustomerID, 
    COUNT(OrderID) AS OrderCount
FROM Orders
GROUP BY CustomerID
HAVING COUNT(OrderID) > 3;

5. What are the top 5 most expensive products? Round the price to 2 decimal places.
SELECT 
    ProductName, 
    ROUND(CAST(Price AS NUMERIC), 2) AS Price
FROM Products
ORDER BY Price DESC
LIMIT 5;

6. What are the order details (ProductID, Quantity) for customers from France?
SELECT 
	o.OrderID, 
	c.CustomerName, 
	c.Country, 
	od.ProductID, 
	od.Quantity
FROM Orders o
JOIN Customers c ON o.CustomerID = c.CustomerID
JOIN OrderDetails od ON o.OrderID = od.OrderID
WHERE c.Country = 'France';

7. Area there products without a category assigned?
SELECT 
    ProductID, 
    ProductName,
    CategoryID
FROM Products
WHERE CategoryID IS NULL;

8. What are all orders and their employees?
SELECT 
    o.OrderID, 
    e.EmployeeID, 
    e.FirstName, 
    e.LastName
FROM Orders o
LEFT JOIN Employees e ON o.EmployeeID = e.EmployeeID;

9. What is the average, minimum, and maximum price of products? Round the values to 2 decimal places.
SELECT 
    ROUND(CAST(AVG(Price) AS NUMERIC), 2) AS AveragePrice,
    ROUND(CAST(MIN(Price) AS NUMERIC), 2) AS MinimumPrice,
    ROUND(CAST(MAX(Price) AS NUMERIC), 2) AS MaximumPrice
FROM Products;

10. What are the products with prices between 10 and 50? Round the price to 2 decimal places and sort the result by price in descending order.
SELECT 
    ProductID, 
    ProductName, 
    ROUND(CAST(Price AS NUMERIC), 2) AS Price
FROM Products
WHERE Price BETWEEN 10 AND 50
ORDER BY Price DESC;

11. What are the shippers and the total number of orders shipped by each shipper, including those with no orders?
SELECT 
    s.ShipperName, 
    COUNT(o.OrderID) AS TotalOrders
FROM Shippers s
LEFT JOIN Orders o ON s.ShipperID = o.ShipperID
GROUP BY s.ShipperName;

12. What are the employees who have processed > 5 orders? Sort the result by the number of orders in descending order.
SELECT 
	e.EmployeeID, 
	e.FirstName, 
	e.LastName, 
COUNT(o.OrderID) AS OrderCount
FROM Employees e
JOIN Orders o ON e.EmployeeID = o.EmployeeID
GROUP BY e.EmployeeID, e.FirstName, e.LastName
HAVING COUNT(o.OrderID) > 5
ORDER BY OrderCount DESC;

13. What is the total revenue for each product within each order, including the product name and ordered by order ID and total revenue in descending order?
SELECT 
	od.OrderID,
	p.ProductID,
	p.ProductName,
	ROUND(CAST(SUM(p.Price * od.Quantity) AS NUMERIC), 2) AS TotalRevenue
FROM OrderDetails od
JOIN Products p ON od.ProductID = p.ProductID
GROUP BY od.OrderID, p.ProductID, p.ProductName
ORDER BY od.OrderID, TotalRevenue DESC;

14. What are the customers, employees, and the total number of orders placed by each customer?
SELECT 
    c.CustomerID, 
    c.CustomerName, 
    e.EmployeeID, 
    e.FirstName, 
    e.LastName, 
    COUNT(o.OrderID) AS TotalOrders
FROM Orders o
JOIN Customers c ON o.CustomerID = c.CustomerID
JOIN Employees e ON o.EmployeeID = e.EmployeeID
GROUP BY c.CustomerID, c.CustomerName, e.EmployeeID, e.FirstName, e.LastName;


15. What are the products with an average price higher than the overall average product price? 
    Round the price to 2 decimal places and sort the result by price in descending order.
SELECT 
    ProductID, 
    ProductName, 
    ROUND(CAST(Price AS NUMERIC), 2) AS Price
FROM Products
WHERE Price > (SELECT AVG(Price) FROM Products)
ORDER BY Price DESC;
